# PenuX x AmsterdamUMCdb get-data notebook

Run after official AmsterdamUMCdb access is granted. This notebook clones the public helper repo, loads OMOP v1.5 tables from local files or BigQuery, and exports early 6h/12h summary features for PenuX. Do not commit generated patient-level outputs.


In [ ]:
# Optional setup
# !pip -q install pandas numpy pyarrow pandas-gbq google-cloud-bigquery
# !pip -q install git+https://github.com/AmsterdamUMC/AmsterdamUMCdb.git
from pathlib import Path
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
DATA_ROOT=Path(os.environ.get('AUMCDB_ROOT','data/amsterdamumcdb'))
OUT_DIR=Path(os.environ.get('PENUX_OUT_DIR','data/processed'))
OUT_DIR.mkdir(parents=True, exist_ok=True)
WINDOWS=[6,12]
if not Path('external/AmsterdamUMCdb').exists():
    Path('external').mkdir(exist_ok=True)
    !git clone --depth 1 https://github.com/AmsterdamUMC/AmsterdamUMCdb.git external/AmsterdamUMCdb
def find_table(root, table):
    for r in [root,root/'omop',root/'OMOP',root/'csv',root/'parquet']:
        for n in [table,table.upper(),table.lower()]:
            for s in ['.parquet','.csv.gz','.csv']:
                p=r/f'{n}{s}'
                if p.exists(): return p
    return None
def load_local(table, cols=None):
    p=find_table(DATA_ROOT, table)
    if p is None: raise FileNotFoundError(table)
    print('Loading', table, p)
    return pd.read_parquet(p, columns=cols) if p.name.endswith('.parquet') else pd.read_csv(p, usecols=cols, low_memory=False)
USE_BQ=bool(os.environ.get('AUMCDB_BQ_DATASET'))
BQ=os.environ.get('AUMCDB_BQ_DATASET','')
PROJ=os.environ.get('GOOGLE_CLOUD_PROJECT','')
if USE_BQ:
    import pandas_gbq
    def bq(sql): return pandas_gbq.read_gbq(sql, project_id=PROJ, dialect='standard')
else:
    def bq(sql): raise RuntimeError('Set AUMCDB_BQ_DATASET and GOOGLE_CLOUD_PROJECT')
print('DATA_ROOT', DATA_ROOT.resolve(), 'USE_BQ', USE_BQ, 'BQ', BQ)


In [ ]:
person = bq(f'SELECT * FROM `{BQ}.person`') if USE_BQ else load_local('person')
visits = bq(f'SELECT * FROM `{BQ}.visit_occurrence`') if USE_BQ else load_local('visit_occurrence')
concept = bq(f'SELECT concept_id, concept_name FROM `{BQ}.concept`') if USE_BQ else load_local('concept')
def pick(df, names):
    m={c.lower():c for c in df.columns}
    return next((m[n.lower()] for n in names if n.lower() in m), None)
vid=pick(visits,['visit_occurrence_id','visit_id','admissionid']); pid=pick(visits,['person_id','patientid']); start=pick(visits,['visit_start_datetime','admittedat','admissiontime','startdate'])
assert vid and pid and start
cohort=visits[[vid,pid,start]].copy().rename(columns={vid:'visit_id',pid:'person_id',start:'t0'})
cohort['t0']=pd.to_datetime(cohort['t0'],errors='coerce',utc=True).dt.tz_convert(None)
cohort=cohort.dropna(subset=['visit_id','person_id','t0']).drop_duplicates('visit_id')
yob=pick(person,['year_of_birth']); gender=pick(person,['gender_concept_id','gender_source_value','sex','gender']); ppid=pick(person,['person_id','patientid'])
demo=person[[ppid]+([yob] if yob else [])+([gender] if gender else [])].copy().rename(columns={ppid:'person_id'})
demo['year_of_birth']=pd.to_numeric(demo[yob],errors='coerce') if yob else np.nan
demo['gender_raw']=demo[gender].astype(str) if gender else 'UNKNOWN'
X=cohort.merge(demo[['person_id','year_of_birth','gender_raw']],on='person_id',how='left')
X['age_at_t0']=X['t0'].dt.year-pd.to_numeric(X['year_of_birth'],errors='coerce')
X['gender']=X['gender_raw'].astype(str).fillna('UNKNOWN')
print('cohort', X.shape)


In [ ]:
TERMS={'heart_rate':'heart rate|pulse','resp_rate':'respiratory rate|respiration rate','spo2':'oxygen saturation|spo2','temperature':'temperature','systolic_bp':'systolic blood pressure|systolic arterial pressure','diastolic_bp':'diastolic blood pressure|diastolic arterial pressure','mean_bp':'mean arterial pressure|mean blood pressure','wbc':'white blood cell|leukocyte|wbc','creatinine':'creatinine','bun':'blood urea nitrogen|urea nitrogen|ureum|urea','lactate':'lactate','sodium':'sodium','potassium':'potassium','chloride':'chloride','bicarbonate':'bicarbonate|hco3','bilirubin':'bilirubin','platelets':'platelet','hemoglobin':'hemoglobin|haemoglobin','crp':'c-reactive protein|crp'}
cid=pick(concept,['concept_id']); cname=pick(concept,['concept_name']); concept['_n']=concept[cname].astype(str).str.lower()
concept_map={}
for k,pat in TERMS.items():
    ids=concept.loc[concept['_n'].str.contains(pat,case=False,regex=True,na=False),cid].dropna().astype(int).unique().tolist()
    concept_map[k]=ids; print(k,len(ids),ids[:5])
sel=sorted({i for v in concept_map.values() for i in v})
cols=['measurement_id','person_id','visit_occurrence_id','measurement_concept_id','measurement_datetime','value_as_number','unit_concept_id']
if USE_BQ:
    ids=','.join(map(str,sel)) or '-1'
    meas=bq(f'''SELECT measurement_id, person_id, visit_occurrence_id, measurement_concept_id, measurement_datetime, value_as_number, unit_concept_id FROM `{BQ}.measurement` WHERE measurement_concept_id IN ({ids}) AND value_as_number IS NOT NULL''')
else:
    mf=find_table(DATA_ROOT,'measurement')
    if mf.name.endswith('.parquet'):
        meas=pd.read_parquet(mf, columns=[c for c in cols if c in pd.read_parquet(mf, columns=[]).columns])
        meas=meas[meas['measurement_concept_id'].isin(sel)]
    else:
        parts=[]
        for ch in pd.read_csv(mf,usecols=lambda c:c in cols,chunksize=1000000,low_memory=False):
            ch=ch[ch['measurement_concept_id'].isin(sel)]
            if len(ch): parts.append(ch)
        meas=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=cols)
id2f={i:k for k,ids in concept_map.items() for i in ids}
meas=meas.rename(columns={'visit_occurrence_id':'visit_id'})
meas['feature_name']=meas['measurement_concept_id'].map(id2f)
meas['measurement_datetime']=pd.to_datetime(meas['measurement_datetime'],errors='coerce',utc=True).dt.tz_convert(None)
meas['value_as_number']=pd.to_numeric(meas['value_as_number'],errors='coerce')
m=meas.dropna(subset=['visit_id','feature_name','measurement_datetime','value_as_number']).merge(cohort[['visit_id','t0']],on='visit_id',how='inner')
m['h']=(m['measurement_datetime']-m['t0']).dt.total_seconds()/3600
m=m[(m['h']>=0)&(m['h']<=max(WINDOWS))]
print('early rows',m.shape)
def summ(g):
    g=g.sort_values('h'); x=g['h'].to_numpy(float); y=g['value_as_number'].to_numpy(float)
    return pd.Series({'mean':np.nanmean(y),'std':np.nanstd(y),'min':np.nanmin(y),'max':np.nanmax(y),'first':y[0],'last':y[-1],'delta':(y[-1]-y[0]) if len(y)>1 else 0.0,'slope':float(np.polyfit(x,y,1)[0]) if len(y)>1 and np.nanstd(x)>0 else 0.0,'count':len(y)})
F=X[['visit_id','person_id','t0','age_at_t0','gender']].copy()
for wh in WINDOWS:
    s=m[m['h']<=wh].groupby(['visit_id','feature_name']).apply(summ).reset_index()
    if len(s):
        w=s.pivot(index='visit_id',columns='feature_name'); w.columns=[f'{feat}_w{wh}h_{stat}' for stat,feat in w.columns]; F=F.merge(w.reset_index(),on='visit_id',how='left')
print('feature matrix',F.shape)
OUT_DIR.mkdir(parents=True, exist_ok=True)
F.to_parquet(OUT_DIR/'amsterdamumcdb_penux_omop_features.parquet',index=False)
F.to_csv(OUT_DIR/'amsterdamumcdb_penux_omop_features.csv.gz',index=False,compression='gzip')
F.head()
